# Crime Prediction in New York using ConvLSTM and Geospatial Heatmaps

## Overview

This project explores spatiotemporal crime forecasting in New York City using deep learning and geospatial analysis.

The pipeline:
1. Cleans and preprocesses NYC crime incident data
2. Converts geographic coordinates into spatial grid cells
3. Builds temporal occupancy tensors
4. Trains a ConvLSTM neural network
5. Predicts future crime hotspot activity across New York
6. Visualizes predictions using animated GIS heatmaps

The final model is deployed through an interactive Streamlit application for dynamic crime hotspot forecasting. 

## Library Imports

The following libraries are used throughout the project for:

- data preprocessing and manipulation
- numerical computation
- geospatial visualization
- deep learning model development
- temporal forecasting
- interactive dashboard deployment

TensorFlow and Keras are used to build the ConvLSTM architecture for spatiotemporal crime prediction across New York City.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, BatchNormalization, Conv2D
from tensorflow.keras.optimizers import Adam

## Data Loading

The dataset used in this project contains historical NYPD complaint records across New York City.

The data includes:
- geographic coordinates
- timestamps
- offense information
- borough-level crime reports

The dataset is loaded into a Pandas DataFrame for preprocessing and spatial-temporal feature engineering.

In [2]:
path_to_file = "/kaggle/input/datasets/brunacmendes/nypd-complaint-data-historic-20062019/NYPD_Complaint_Data_Historic.csv"
df = pd.read_csv(path_to_file)

/tmp/ipykernel_55/2144408616.py:3: DtypeWarning: Columns (18,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path_to_file)


In [3]:
df

,CMPLNT_NUM,CMPLNT_FR_DT,CMPLNT_FR_TM,CMPLNT_TO_DT,CMPLNT_TO_TM,ADDR_PCT_CD,RPT_DT,KY_CD,OFNS_DESC,PD_CD,...,SUSP_SEX,TRANSIT_DISTRICT,Latitude,Longitude,Lat_Lon,PATROL_BORO,STATION_NAME,VIC_AGE_GROUP,VIC_RACE,VIC_SEX
0,700381962,05/28/2015,15:00:00,NaN,NaN,46.0,06/01/2015,578,HARRASSMENT 2,638.0,...,M,NaN,40.845868,-73.915888,"(40.84586773, -73.915888033)",PATROL BORO BRONX,NaN,25-44,WHITE HISPANIC,F
1,642234217,10/28/2013,13:50:00,10/28/2013,13:50:00,120.0,10/28/2013,351,CRIMINAL MISCHIEF & RELATED OF,259.0,...,NaN,NaN,40.627061,-74.077149,"(40.627060894, -74.077149232)",PATROL BORO STATEN ISLAND,NaN,45-64,WHITE,M
2,242465164,05/09/2012,20:50:00,05/09/2012,21:00:00,24.0,05/09/2012,236,DANGEROUS WEAPONS,782.0,...,NaN,NaN,40.800966,-73.969047,"(40.800965968, -73.969047272)",PATROL BORO MAN NORTH,NaN,NaN,UNKNOWN,E
3,927207428,01/03/2014,13:30:00,01/03/2014,13:35:00,108.0,01/03/2014,109,GRAND LARCENY,409.0,...,M,NaN,40.745242,-73.894253,"(40.745241809, -73.894253382)",PATROL BORO QUEENS NORTH,NaN,45-64,ASIAN / PACIFIC ISLANDER,M
4,492142357,04/13/2016,00:00:00,NaN,NaN,40.0,04/13/2016,351,CRIMINAL MISCHIEF & RELATED OF,258.0,...,NaN,NaN,40.810352,-73.924942,"(40.810351863, -73.924942326)",PATROL BORO BRONX,NaN,UNKNOWN,UNKNOWN,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6983202,225471008,12/02/2018,04:15:00,12/02/2018,04:17:00,109.0,12/02/2018,344,ASSAULT 3 & RELATED OFFENSES,101.0,...,M,NaN,40.785050,-73.856852,"(40.785049616, -73.856851768)",PATROL BORO QUEENS NORTH,NaN,25-44,ASIAN / PACIFIC ISLANDER,F
6983203,368441925,01/20/2018,00:08:00,04/26/2019,12:00:00,25.0,04/26/2019,361,OFF. AGNST PUB ORD SENSBLTY &,639.0,...,F,NaN,NaN,NaN,NaN,PATROL BORO MAN NORTH,NaN,25-44,BLACK,M
6983204,146134182,08/03/2018,22:30:00,08/04/2018,06:30:00,41.0,08/04/2018,121,CRIMINAL MISCHIEF & RELATED OF,267.0,...,NaN,NaN,40.814612,-73.903637,"(40.814612305, -73.903637247)",PATROL BORO BRONX,NaN,25-44,BLACK HISPANIC,F
6983205,763119484,12/10/2018,11:00:00,12/21/2018,15:00:00,107.0,04/10/2019,340,FRAUDS,718.0,...,U,NaN,NaN,NaN,NaN,PATROL BORO QUEENS SOUTH,NaN,18-24,ASIAN / PACIFIC ISLANDER,M


## Data Cleaning and Temporal Feature Engineering

To prepare the dataset for spatiotemporal modeling:

- only the relevant spatial and temporal columns are retained
- rows with missing geographic information are removed
- date and time columns are combined into a unified timestamp
- invalid or malformed timestamps are discarded

This preprocessing step creates a clean temporal structure required for sequential crime forecasting.

In [5]:
df = df[['CMPLNT_FR_DT','CMPLNT_FR_TM','Latitude','Longitude']]
df = df.dropna()

df['datetime'] = pd.to_datetime(
    df['CMPLNT_FR_DT'] + ' ' + df['CMPLNT_FR_TM'],
    format='%m/%d/%Y %H:%M:%S',
    errors='coerce'
)
df = df.dropna(subset=['datetime']) # null vals still present

## Geographic Boundary Filtering

To ensure spatial consistency, crime incidents are restricted to the geographic boundaries of New York City.

Records with coordinates outside the NYC area are removed.  
This helps eliminate:
- invalid geographic points
- misplaced coordinates
- outlier locations outside the city limits

The remaining data represents crime activity occurring within New York City's spatial boundaries.

In [7]:
NYC_contours = {
    'lat_min':40.49,
    'lat_max':40.92,
    'lon_min':-74.26,
    'lon_max':-73.70
}

df = df[
(df['Latitude']>=NYC_contours['lat_min']) &
(df['Latitude']<=NYC_contours['lat_max']) &
(df['Longitude']>=NYC_contours['lon_min']) &
(df['Longitude']<=NYC_contours['lon_max'])
]

## Spatial Grid Construction

To model crime activity spatially, New York City is divided into a fixed geographic grid.

Latitude and longitude coordinates are discretized into spatial bins, allowing each crime incident to be mapped to a grid cell.

This transformation converts raw geographic coordinates into a structured spatial representation suitable for ConvLSTM processing.

In [8]:
GRID_SIZE = 24

lat_bins = np.linspace(NYC_contours['lat_min'],NYC_contours['lat_max'],GRID_SIZE+1)
lon_bins = np.linspace(NYC_contours['lon_min'],NYC_contours['lon_max'],GRID_SIZE+1)

df['lat_bin'] = pd.cut(df['Latitude'], bins=lat_bins, labels=False)
df['lon_bin'] = pd.cut(df['Longitude'], bins=lon_bins, labels=False)

In [9]:
#dates sorted ascendingly
df = df.sort_values(by="datetime").reset_index(drop=True)

In [10]:
df

,CMPLNT_FR_DT,CMPLNT_FR_TM,Latitude,Longitude,datetime,lat_bin,lon_bin
0,03/10/1900,19:00:00,40.646130,-73.973227,1900-03-10 19:00:00,8,12
1,05/08/1900,21:00:00,40.788861,-73.952004,1900-05-08 21:00:00,16,13
2,08/06/1900,09:00:00,40.774602,-73.945017,1900-08-06 09:00:00,15,13
3,08/07/1900,08:30:00,40.603887,-73.942543,1900-08-07 08:30:00,6,13
4,11/26/1900,19:00:00,40.735323,-73.984240,1900-11-26 19:00:00,13,11
...,...,...,...,...,...,...,...
6958172,12/31/2019,23:20:00,40.761752,-73.983840,2019-12-31 23:20:00,15,11
6958173,12/31/2019,23:20:00,40.789947,-73.975354,2019-12-31 23:20:00,16,12
6958174,12/31/2019,23:21:00,40.694098,-73.934269,2019-12-31 23:21:00,11,13
6958175,12/31/2019,23:25:00,40.686729,-73.952712,2019-12-31 23:25:00,10,13


## Temporal Tensor Generation

To prepare the dataset for deep learning, crime activity is transformed into a spatiotemporal tensor representation.

For each day:
- crime incidents are grouped by spatial grid cell
- duplicate occurrences within the same cell are removed
- a binary occupancy grid is created

This produces a 3D tensor where:

- time represents daily crime evolution
- latitude bins represent vertical spatial regions
- longitude bins represent horizontal spatial regions

The resulting tensor becomes the input structure for ConvLSTM forecasting.

In [12]:
df['day'] = df['datetime'].dt.to_period('D')

# keep unique (week, cell) combinations
crime_cells = df[['day','lat_bin','lon_bin']].drop_duplicates()

time_periods = sorted(crime_cells['day'].unique())
n_periods = len(time_periods)

crime_tensor = np.zeros((n_periods, GRID_SIZE, GRID_SIZE))

period_to_idx = {p:i for i,p in enumerate(time_periods)}

for _, row in crime_cells.iterrows():
    
    t = period_to_idx[row['day']]
    x = int(row['lat_bin'])
    y = int(row['lon_bin'])
    
    crime_tensor[t, x, y] = 1

/tmp/ipykernel_55/1986105497.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['day'] = df['datetime'].dt.to_period('D')


In [13]:
df

,CMPLNT_FR_DT,CMPLNT_FR_TM,Latitude,Longitude,datetime,lat_bin,lon_bin,day
1000000,11/13/2007,10:30:00,40.734572,-73.889828,2007-11-13 10:30:00,13,15,2007-11-13
1000001,11/13/2007,10:30:00,40.697496,-73.933631,2007-11-13 10:30:00,11,13,2007-11-13
1000002,11/13/2007,10:30:00,40.730517,-73.866177,2007-11-13 10:30:00,13,16,2007-11-13
1000003,11/13/2007,10:30:00,40.629666,-74.028531,2007-11-13 10:30:00,7,9,2007-11-13
1000004,11/13/2007,10:30:00,40.818741,-73.914253,2007-11-13 10:30:00,18,14,2007-11-13
...,...,...,...,...,...,...,...,...
6958172,12/31/2019,23:20:00,40.761752,-73.983840,2019-12-31 23:20:00,15,11,2019-12-31
6958173,12/31/2019,23:20:00,40.789947,-73.975354,2019-12-31 23:20:00,16,12,2019-12-31
6958174,12/31/2019,23:21:00,40.694098,-73.934269,2019-12-31 23:21:00,11,13,2019-12-31
6958175,12/31/2019,23:25:00,40.686729,-73.952712,2019-12-31 23:25:00,10,13,2019-12-31


In [14]:
#processed data saved to be used as input for the trained model
df.to_csv("processed_data.csv", index=False)

## ConvLSTM Hotspot Prediction Pipeline

This section builds the complete deep learning pipeline for crime hotspot forecasting in New York City.

The workflow includes:
- converting crime activity into binary hotspot representations
- generating temporal training sequences
- splitting the dataset into training, validation, and testing subsets
- constructing a ConvLSTM neural network for spatiotemporal forecasting
- training the model using weighted hotspot classification
- evaluating predictive performance using precision, recall, and F1-score
- exporting the trained model for deployment in the Streamlit application

The ConvLSTM architecture enables the model to learn both spatial crime distributions and temporal crime evolution patterns across NYC grid cells.

In [17]:
# Hotspot ConvLSTM pipeline 

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import ConvLSTM2D, Conv2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

# =====  Convert crime counts to binary hotspots =====
# y_tensor = your normalized or raw crime tensor (time, GRID_SIZE, GRID_SIZE)
# any crime = hotspot
y_binary_tensor = crime_tensor # (crime_tensor > 0).astype(int)  # shape: (time, GRID_SIZE, GRID_SIZE)



# ===== 2️ Create sequences =====
TIME_STEPS = 24  # past 24 days to predict next
GRID_SIZE = crime_tensor.shape[1]

def create_hotspot_sequences(data, n_steps):
    X, y = [], []
    for i in range(len(data) - n_steps):
        seq_x = data[i:i+n_steps]       # input sequence
        seq_y = data[i+n_steps]         # target hotspot map
        X.append(seq_x)
        y.append(seq_y)
    X = np.array(X)[..., np.newaxis]  # add channel dimension
    y = np.array(y)[..., np.newaxis]
    return X, y

X, y_binary = create_hotspot_sequences(y_binary_tensor, TIME_STEPS)
print(f"Sequences created: X={X.shape}, y={y_binary.shape}")

# ===== 3️ Train/validation/test split =====
X_train, X_temp, y_train, y_temp = train_test_split(X, y_binary, test_size=0.3, shuffle=False)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, shuffle=False)

print(f"Training: {X_train.shape[0]} samples, Validation: {X_val.shape[0]}, Test: {X_test.shape[0]}")

# ===== 4️ Build ConvLSTM hotspot model =====
input_shape = (TIME_STEPS, GRID_SIZE, GRID_SIZE, 1)

model = Sequential([
    
    ConvLSTM2D(64, (3,3), padding='same', return_sequences=True, activation='relu', input_shape=input_shape),
    BatchNormalization(),
    ConvLSTM2D(16, (3,3), padding='same', return_sequences=False, activation='relu'),
    BatchNormalization(),
    Conv2D(1, (1,1), activation='sigmoid', padding='same')  # probability of hotspot
])

model.compile(optimizer=Adam(0.01), loss='binary_crossentropy', metrics=['precision'])
model.summary()

# Compute hotspot / non-hotspot imbalance
pos = np.sum(y_train)
neg = np.prod(y_train.shape) - pos

# Increase weight of hotspot cells during training
# to reduce class imbalance effects
pos_weight = neg / (pos) * 0.25

# Assign higher weights to hotspot locations
sample_weights = np.where(
    y_train == 1,
    pos_weight,
    1.0
)

# ===== Train model =====
history = model.fit(
    X_train, y_train,
    sample_weight = sample_weights,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=8
)

# ===== Predict and evaluate =====
y_pred_prob = model.predict(X_test)
y_pred_binary = (y_pred_prob > 0.1).astype(int)

# Flatten for metrics
y_test_flat = y_test.flatten()
y_pred_flat = y_pred_binary.flatten()

precision = precision_score(y_test_flat, y_pred_flat)
recall = recall_score(y_test_flat, y_pred_flat)
f1 = f1_score(y_test_flat, y_pred_flat)

print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-score: {f1:.4f}")
model.save('model.keras')

Sequences created: X=(4408, 24, 24, 24, 1), y=(4408, 24, 24, 1)
Training: 3085 samples, Validation: 661, Test: 662


I0000 00:00:1775376602.010675      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv_lstm2d (ConvLSTM2D)        │ (None, 24, 24, 24, 64) │       150,016 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 24, 24, 24, 64) │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv_lstm2d_1 (ConvLSTM2D)      │ (None, 24, 24, 16)     │        46,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 16)     │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 24, 24, 1)      │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 196,497 (767.57 KB)

...

Epoch 1/20


I0000 00:00:1775376608.019413     122 service.cc:152] XLA service 0x423906b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775376608.019452     122 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1775376608.915542     122 cuda_dnn.cc:529] Loaded cuDNN version 91002


  3/386 ━━━━━━━━━━━━━━━━━━━━ 21s 55ms/step - loss: 0.7852 - precision: 0.2916

I0000 00:00:1775376612.778253     122 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


386/386 ━━━━━━━━━━━━━━━━━━━━ 36s 71ms/step - loss: 0.3746 - precision: 0.7294 - val_loss: 0.7784 - val_precision: 0.0000e+00
Epoch 2/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 22s 56ms/step - loss: 0.1469 - precision: 0.8951 - val_loss: 0.8079 - val_precision: 0.0000e+00
Epoch 3/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 22s 56ms/step - loss: 0.1399 - precision: 0.8959 - val_loss: 0.8340 - val_precision: 0.0000e+00
Epoch 4/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 22s 56ms/step - loss: 0.1327 - precision: 0.9074 - val_loss: 0.8674 - val_precision: 0.0000e+00
Epoch 5/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 22s 57ms/step - loss: 0.1288 - precision: 0.9115 - val_loss: 0.8585 - val_precision: 0.0000e+00
Epoch 6/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 22s 56ms/step - loss: 0.1272 - precision: 0.9143 - val_loss: 0.9063 - val_precision: 0.0000e+00
Epoch 7/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 22s 57ms/step - loss: 0.1242 - precision: 0.9165 - val_loss: 0.8759 - val_precision: 0.0000e+00
Epoch 8/20
386/386 ━━━━━━━━━━━━━━━━━━━━ 22s 57ms/step - los